In [ ]:
!pip install pandas openpyxl pydub tqdm
!apt-get install ffmpeg -y

import pandas as pd
import re
from pydub import AudioSegment
from tqdm import tqdm
import shutil
import os
import requests
import zipfile
import os
import shutil
from google.colab import files

YANDEX_LINK = "YANDEX LINK"

def download_yandex_disk(url, output):
    api_url = "https://cloud-api.yandex.net/v1/disk/public/resources/download"
    response = requests.get(api_url, params={"public_key": url})
    download_url = response.json()["href"]

    with requests.get(download_url, stream=True) as r:
        with open(output, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)

download_yandex_disk(YANDEX_LINK, "data.zip")

with zipfile.ZipFile("data.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

INPUT_DIR = "data/labeled"
OUTPUT_DIR = "1_urmi_labeled"

os.makedirs(f"{OUTPUT_DIR}/audios", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/texts", exist_ok=True)

LOG_FILE = f"{OUTPUT_DIR}/unlabeled_files.txt"

open(LOG_FILE, "w").close()

def has_russian(text):
    return bool(re.search(r"[а-яА-Я]", str(text)))

def is_timecoded(df):
    if "timecode" not in df.columns:
        return False

    tc_series = df["timecode"].dropna().astype(str).str.strip()

    if tc_series.empty:
        return False

    if len(tc_series) == 1:
        sample = tc_series.iloc[0]
        if sample == "0:00-" or "-" not in sample:
            return False

    return True

def parse_timecode(tc, audio_length_ms):
    if not isinstance(tc, str) or "-" not in tc:
        return None

    start, end = tc.split("-")

    start = start.strip()
    end = end.strip()

    def to_ms(t):
        t = t.replace(",", ".").strip()

        if ":" in t:
            parts = list(map(float, t.split(":")))
            if len(parts) == 2:
                return (parts[0]*60 + parts[1]) * 1000
            elif len(parts) == 3:
                return (parts[0]*3600 + parts[1]*60 + parts[2]) * 1000

        return float(t)

    try:
        start_ms = to_ms(start)
    except:
        return None

    if end == "":
        end_ms = audio_length_ms
    else:
        try:
            end_ms = to_ms(end)
        except:
            return None

    if end_ms <= start_ms:
        return None

    return int(start_ms), int(end_ms)

DETAILED_LOG = f"{OUTPUT_DIR}/detailed_log.csv"

detailed_rows = []

audios_dir = os.path.join(INPUT_DIR, "audios")
texts_dir = os.path.join(INPUT_DIR, "texts")

stats = {
    "total_rows": 0,
    "used": 0,
    "skipped_na_text": 0,
    "skipped_empty_text": 0,
    "skipped_russian_filter": 0,
    "skipped_empty_tc": 0,
    "skipped_bad_tc": 0,
    "unlabeled_files": 0
}

unlabeled_stats = {
    "total_rows": 0,
    "used": 0,
    "skipped_na_text": 0,
    "skipped_empty_text": 0,
    "skipped_russian_filter": 0,
}

for text_file in tqdm(os.listdir(texts_dir)):
    base_name = os.path.splitext(text_file)[0]

    match = re.search(r'\d+', base_name)
    if not match:
        continue
    num = match.group()

    pattern = rf"audio{num}(?!\d)"

    matching_audios = [f for f in os.listdir(audios_dir) if re.search(pattern, f)]

    if not matching_audios:
        print(f"Предупреждение: Аудио для файла {text_file} не найдено (искали номер {num})")
        continue

    audio_file = matching_audios[0]

    text_path = os.path.join(texts_dir, text_file)
    audio_path = os.path.join(audios_dir, audio_file)


    if text_file.endswith(".txt"):
        shutil.copy(text_path, f"{OUTPUT_DIR}/texts/{text_file}")
        shutil.copy(audio_path, f"{OUTPUT_DIR}/audios/{audio_file}")
        continue

    df = pd.read_excel(text_path)

    if not is_timecoded(df):

        unlabeled_stats["total_rows"] += len(df)
        stats["unlabeled_files"] += 1

        valid_texts = []

        for idx, row in df.iterrows():
            raw_text = row["text"]

            reason = None

            if pd.isna(raw_text):
                unlabeled_stats["skipped_na_text"] += 1
                reason = "text_is_nan"

            else:
                text = str(raw_text).strip()

                if not text:
                    unlabeled_stats["skipped_empty_text"] += 1
                    reason = "empty_text"

                elif has_russian(text):
                    unlabeled_stats["skipped_russian_filter"] += 1
                    reason = "russian_text"

            if reason:
                detailed_rows.append({
                    "file": text_file,
                    "row_index": idx,
                    "text": str(raw_text),
                    "timecode": "",
                    "reason": f"unlabeled_{reason}"
                })
                continue

            valid_texts.append(text)
            unlabeled_stats["used"] += 1

        all_text = "\n".join(valid_texts)

        txt_name = f"text{num}_urmi_labeled.txt"

        with open(f"{OUTPUT_DIR}/texts/{txt_name}", "w", encoding="utf-8") as f:
            f.write(all_text.strip())

        shutil.copy(audio_path, f"{OUTPUT_DIR}/audios/{audio_file}")

        with open(LOG_FILE, "a", encoding="utf-8") as log:
            log.write(f"{text_file} -> {txt_name} | total: {len(df)} | used: {len(valid_texts)}\n")

        continue

    stats["total_rows"] += len(df)
    audio = AudioSegment.from_file(audio_path)

    counter = 1

    for idx, row in df.iterrows():

        tc = row["timecode"]
        raw_text = row["text"]

        reason = None

        # текст
        if pd.isna(raw_text):
            stats["skipped_na_text"] += 1
            reason = "text_is_nan"

        else:
            text = str(raw_text).strip()

            if not text:
                stats["skipped_empty_text"] += 1
                reason = "empty_text"

            elif has_russian(text):
                stats["skipped_russian_filter"] += 1
                reason = "russian_text"

        if reason:
            detailed_rows.append({
                "file": text_file,
                "row_index": idx,
                "text": str(raw_text),
                "timecode": str(tc),
                "reason": reason
            })
            continue

        # таймкод
        if pd.isna(tc) or not str(tc).strip():
            stats["skipped_empty_tc"] += 1
            reason = "empty_timecode"

            detailed_rows.append({
                "file": text_file,
                "row_index": idx,
                "text": text,
                "timecode": str(tc),
                "reason": reason
            })
            continue

        tc = str(tc).strip()

        result = parse_timecode(tc, len(audio))

        if result is None:
            stats["skipped_bad_tc"] += 1
            reason = "bad_timecode"

            detailed_rows.append({
                "file": text_file,
                "row_index": idx,
                "text": text,
                "timecode": tc,
                "reason": reason
            })
            continue

        start, end = result
        stats["used"] += 1

        # TXT
        txt_name = f"text{num}-{counter}_urmi_labeled.txt"
        with open(f"{OUTPUT_DIR}/texts/{txt_name}", "w", encoding="utf-8") as f:
            f.write(text)

        # AUDIO
        chunk = audio[start:end]
        audio_name = f"audio{num}-{counter}_urmi_labeled.wav"
        chunk.export(f"{OUTPUT_DIR}/audios/{audio_name}", format="wav")

        counter += 1

log_df = pd.DataFrame(detailed_rows)
log_df.to_csv(DETAILED_LOG, index=False, encoding="utf-8")

print("\nСТАТИСТИКА")
for k, v in stats.items():
    print(f"{k}: {v}")

print("\nUNLABELED СТАТИСТИКА")
for k, v in unlabeled_stats.items():
    print(f"{k}: {v}")

print(f"\nДетальный лог сохранён: {DETAILED_LOG}")

print("\nПРОВЕРКА ПУСТЫХ TXT")

empty_txt_files = []

texts_output_dir = f"{OUTPUT_DIR}/texts"

for fname in os.listdir(texts_output_dir):
    if not fname.endswith(".txt"):
        continue

    fpath = os.path.join(texts_output_dir, fname)

    try:
        with open(fpath, "r", encoding="utf-8") as f:
            content = f.read().strip()

            if not content:
                empty_txt_files.append(fname)

    except Exception as e:
        print(f"Ошибка чтения {fname}: {e}")

if empty_txt_files:
    print(f"Найдено пустых txt: {len(empty_txt_files)}")
    for f in empty_txt_files[:10]:
        print(f"  - {f}")

    if len(empty_txt_files) > 10:
        print(f"... и ещё {len(empty_txt_files) - 10}")
else:
    print("Пустых txt файлов нет")

shutil.make_archive("1_urmi_labeled", 'zip', OUTPUT_DIR)

files.download("1_urmi_labeled.zip")